# Exercises XP: Introduction to LLMs

Use this guided notebook to follow the platform instructions step by step. Prefilled cells are ready to run; cells containing **TODO** markers need your input.


## 👩‍🏫👩🏿‍🏫 What you’ll learn
- Understand what Large Language Models (LLMs) can do.
- Review the Transformer architecture and the tokenization pipeline.
- Differentiate between pretraining and fine-tuning.
- Generate text with a pretrained language model.


## 🛠️ What you will create
- Markdown answers describing key NLP concepts.
- Python code that loads GPT-2 (or a similar causal LM) and performs basic tokenization and generation.


> **Learning point**
> Work through the exercises sequentially. Run installation cells only once, then focus on filling each TODO before executing the corresponding code.


## 🌟 Exercise 1 · What are Large Language Models?


### 1.1 Define LLMs

Les **Large Language Models (LLMs)** sont des modèles de deep learning entraînés sur de très grandes quantités de texte (des milliards de mots) pour apprendre les patterns statistiques du langage naturel.

Ils reposent sur l'architecture **Transformer** et sont capables de :
- **Générer du texte** : compléter une phrase, rédiger un article, écrire du code
- **Répondre à des questions** : question-answering sur un document ou en général
- **Résumer** : condenser un texte long en points clés
- **Traduire** : passer d'une langue à une autre
- **Classer** : analyse de sentiment, détection de spam
- **Dialoguer** : chatbots et assistants conversationnels (ex. ChatGPT, Claude)

Ce qui les distingue des modèles NLP classiques :
- **Taille** : des centaines de millions voire des milliards de paramètres
- **Préentraînement général** puis **fine-tuning** sur des tâches spécifiques
- **Apprentissage en contexte** (in-context learning) : ils peuvent résoudre une tâche avec seulement quelques exemples dans le prompt, sans réentraînement

### 1.2 Prefilled · install core libraries
Run once to install `transformers`, `torch`, and supporting utilities exactly as in the enoncé.


In [ ]:
%pip install --quiet transformers matplotlib --upgrade


### 1.3 Load GPT for causal text generation
Reuse the snippet from the platform: declare the model name, tokenizer, and model weights.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

model_name = "gpt2"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForCausalLM.from_pretrained(model_name)

print(f"\nModel '{model_name}' loaded successfully!")
print("""
GPT-2 is a causal language model, meaning it predicts the next word in a sequence.
It has been trained on a diverse dataset and can generate coherent, contextually relevant text.
""")

## 🌟 Exercise 2 · Transformer Architecture and Tokenization


### Qu'est-ce que la tokenisation ?

La **tokenisation** est l'étape qui convertit un texte brut en une séquence de **tokens** (unités de base) que le modèle peut traiter.

Ce n'est pas simplement un découpage mot par mot. Les LLMs modernes utilisent des algorithmes comme **Byte-Pair Encoding (BPE)** ou **WordPiece** qui :
1. Découpent les mots fréquents en sous-unités (ex. "tokenization" → ["token", "ization"])
2. Gardent les mots très fréquents entiers
3. Représentent les caractères rares lettre par lettre

Chaque token est ensuite converti en un **identifiant numérique** (token ID) qui correspond à sa position dans le vocabulaire du modèle. C'est cette séquence d'IDs que le Transformer reçoit en entrée sous forme de vecteurs (embeddings).

In [ ]:
text = "Artificial intelligence is transforming the world of data science."

tokens     = tokenizer.tokenize(text)
token_ids  = tokenizer.convert_tokens_to_ids(tokens)

print(f"Original Text : {text}")
print(f"Tokens        : {tokens}")
print(f"Token IDs     : {token_ids}")

x_label = "Tokens"
y_label = "Token IDs"
title   = "GPT-2 Tokenization – Token IDs per Token"

plt.figure(figsize=(12, 4))
plt.bar(tokens, token_ids, color="skyblue", edgecolor="white")
plt.xlabel(x_label)
plt.ylabel(y_label)
plt.title(title)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 🌟 Exercise 3 · Token IDs and special prefixes


In [ ]:
if 'tokens' not in globals() or 'token_ids' not in globals():
    raise ValueError("Run Exercise 2 to define `tokens` and `token_ids` first.")

print("Token → ID mapping :")
print("-" * 30)
for token, token_id in zip(tokens, token_ids):
    print(f"  {token!r:20s} → {token_id}")

### Que signifie le préfixe `Ġ` dans le vocabulaire de GPT-2 ?

Dans les tokenizers de type **BPE (Byte-Pair Encoding)** utilisés par GPT-2, le caractère `Ġ` (G avec un point au-dessus) est une représentation spéciale d'un **espace précédant un mot**.

Concrètement :
- `"Ġworld"` signifie que ce token est précédé d'un espace dans le texte original → c'est le **début d'un nouveau mot**
- Un token **sans** `Ġ` (ex. `"ing"`) est une **continuation du mot précédent** (suffixe)

Cela permet au tokenizer de reconstituer fidèlement le texte original depuis la séquence de tokens, en sachant exactement où les espaces se trouvaient, sans avoir besoin d'un token séparé pour les espaces.

## 🌟 Exercise 4 · Generate simple text


Create a fresh prompt, run the generator, and observe how the model extends your sentence token by token.


In [ ]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

input_text = "The future of artificial intelligence in healthcare is"

gen_kwargs = {
    "max_new_tokens": 60,
    "temperature": 0.8,
    "top_p": 0.95,
    "do_sample": True,
}

output_ids  = generator(input_text, **gen_kwargs)
output_text = output_ids[0]["generated_text"]

print(f"Input            : {input_text}")
print(f"\nGenerated Output :\n{output_text}")

> **Learning point**
> Compare the generated continuation with your expectations. Which knobs (temperature, max tokens) change the style the most?


Here's a summary that can help you decide of how to fix these parameters:

![image.png](https://github.com/user-attachments/assets/a4c444d7-fab8-4f56-b7c7-00a15900cb5a)